# UAS Analisis Data Kategorik
## Regresi Logistik Multinomial pada Risiko Penyakit Kardiovaskular

Notebook ini dibuat untuk topik **Regresi Logistik Multinomial** menggunakan dataset **CAIR-CVD-2025: An Extensive Cardiovascular Disease Risk Assessment Dataset from Bangladesh**.

**Sumber data:** Md Asraful Sharker Nirob, Prayma Bishshash, A K M Fazlul Kobir Siam, Md. Afzalul Haque, dan Md Assaduzzaman. Dipublikasikan 3 Maret 2025. DOI: `10.17632/d9scg7j8fp.1`.

**File lokal:** `CVD Dataset.csv`  
**Variabel respon:** `CVD Risk Level` dengan kategori `LOW`, `INTERMEDIARY`, dan `HIGH`.


## Ringkasan

Penelitian ini menganalisis faktor-faktor yang berkaitan dengan tingkat risiko penyakit kardiovaskular pada pasien di Bangladesh menggunakan regresi logistik multinomial. Data yang digunakan adalah CAIR-CVD-2025 yang berisi 1.529 observasi pasien dengan variabel demografis, antropometrik, klinis, biokimia, serta gaya hidup. Variabel respon yang dianalisis adalah tingkat risiko CVD dengan tiga kategori, yaitu rendah, menengah, dan tinggi. Variabel prediktor yang digunakan pada model utama dibatasi pada kolom asli dataset tanpa feature engineering, yaitu usia, indeks massa tubuh, lingkar perut, tekanan darah sistolik dan diastolik, total kolesterol, HDL, gula darah puasa, estimasi LDL, berat badan, tinggi badan, rasio lingkar pinggang terhadap tinggi badan, jenis kelamin, status merokok, status diabetes, aktivitas fisik, riwayat keluarga CVD, dan kategori tekanan darah. Analisis dilakukan melalui eksplorasi data, penghapusan baris yang missing hanya pada target atau fitur yang dipakai, pembentukan model regresi logistik multinomial menggunakan seluruh data lengkap yang relevan, evaluasi klasifikasi pada data pembentukan model, dan interpretasi odds ratio. Model menggunakan kategori `LOW` sebagai kategori referensi sehingga koefisien dan odds ratio menjelaskan kecenderungan masuk kategori `INTERMEDIARY` atau `HIGH` dibandingkan risiko rendah. Variabel `CVD Risk Score` tetap tidak digunakan sebagai prediktor utama karena berpotensi menjadi sumber kebocoran informasi terhadap `CVD Risk Level`.

**Kata kunci:** regresi logistik multinomial, penyakit kardiovaskular, odds ratio, klasifikasi, CAIR-CVD-2025.


# BAB I. Pendahuluan

## 1.1 Latar Belakang

Penyakit kardiovaskular masih menjadi salah satu penyebab utama morbiditas dan mortalitas di berbagai negara. Identifikasi dini terhadap kelompok pasien berisiko rendah, menengah, dan tinggi penting untuk mendukung strategi pencegahan, pemantauan klinis, serta edukasi gaya hidup. Dataset CAIR-CVD-2025 menyediakan informasi pasien dari Jamalpur Medical College Hospital, Bangladesh, mencakup faktor demografis, antropometrik, klinis, biokimia, dan gaya hidup yang relevan untuk asesmen risiko CVD.

Karena variabel respon memiliki lebih dari dua kategori yang bersifat nominal/ordinal praktis (`LOW`, `INTERMEDIARY`, `HIGH`), metode regresi logistik multinomial digunakan untuk memodelkan peluang setiap tingkat risiko berdasarkan sekumpulan variabel prediktor.

## 1.2 Perumusan Masalah

1. Bagaimana karakteristik deskriptif pasien berdasarkan variabel klinis dan gaya hidup pada dataset CAIR-CVD-2025?
2. Bagaimana membentuk model regresi logistik multinomial untuk mengklasifikasikan tingkat risiko CVD?
3. Variabel apa saja yang cenderung meningkatkan atau menurunkan odds pasien berada pada kategori `INTERMEDIARY` dan `HIGH` dibandingkan `LOW`?

## 1.3 Tujuan Penelitian

1. Mendeskripsikan karakteristik data pasien dan distribusi tingkat risiko CVD.
2. Membangun model regresi logistik multinomial untuk memprediksi `CVD Risk Level`.
3. Menginterpretasikan koefisien model melalui odds ratio.

## 1.4 Manfaat Penelitian

Hasil analisis dapat membantu memahami faktor yang berasosiasi dengan tingkat risiko CVD sehingga dapat menjadi dasar awal untuk edukasi kesehatan, pemantauan risiko, dan pengembangan model skrining berbasis data.

## 1.5 Batasan Penelitian

Analisis ini bersifat observasional dan menggunakan dataset sekunder. Model tidak dimaksudkan sebagai alat diagnosis klinis final. Variabel `CVD Risk Score` tidak digunakan sebagai prediktor utama karena berpotensi sangat dekat dengan proses pembentukan variabel respon `CVD Risk Level`.


# BAB II. Tinjauan Pustaka Singkat

## 2.1 Regresi Logistik Multinomial

Regresi logistik multinomial digunakan ketika variabel respon memiliki lebih dari dua kategori. Jika terdapat kategori referensi, model membandingkan log odds setiap kategori non-referensi terhadap kategori referensi. Pada penelitian ini, kategori `LOW` dipakai sebagai referensi.

Untuk kategori respon \(j\), bentuk umum model adalah:

\[
\log\left(\frac{P(Y=j)}{P(Y=\text{LOW})}\right) = \beta_{0j} + \beta_{1j}X_1 + \cdots + \beta_{pj}X_p
\]

Nilai \(\exp(\beta)\) diinterpretasikan sebagai odds ratio. Odds ratio lebih dari 1 menunjukkan peningkatan odds terhadap kategori pembanding, sedangkan odds ratio kurang dari 1 menunjukkan penurunan odds.

## 2.2 Risiko Penyakit Kardiovaskular

Risiko CVD dapat dipengaruhi oleh faktor usia, indeks massa tubuh, tekanan darah, kadar kolesterol, gula darah, merokok, diabetes, aktivitas fisik, dan riwayat keluarga. Variabel-variabel tersebut dipakai sebagai prediktor karena relevan secara klinis dan tersedia dalam dataset.


# BAB III. Metodologi Penelitian

## 3.1 Unit Penelitian

Unit penelitian adalah satu pasien pada dataset CAIR-CVD-2025.

## 3.2 Pengambilan Sampel

Dataset terdiri dari 1.529 sampel pasien yang dikumpulkan di Jamalpur Medical College Hospital, Jamalpur, Bangladesh, pada 20 Januari 2024 sampai 1 Januari 2025.

## 3.3 Variabel Penelitian

Variabel respon:

- `CVD Risk Level`: tingkat risiko penyakit kardiovaskular dengan kategori `LOW`, `INTERMEDIARY`, dan `HIGH`.

Variabel prediktor utama dari kolom asli dataset:

- Numerik: `Age`, `BMI`, `Abdominal Circumference (cm)`, `Total Cholesterol (mg/dL)`, `HDL (mg/dL)`, `Fasting Blood Sugar (mg/dL)`, `Systolic BP`, `Diastolic BP`, `Estimated LDL (mg/dL)`, `Weight (kg)`, `Height (m)`, dan `Waist-to-Height Ratio`.
- Kategorik: `Sex`, `Smoking Status`, `Diabetes Status`, `Physical Activity Level`, `Family History of CVD`, dan `Blood Pressure Category`.

Variabel `CVD Risk Score` tetap tidak digunakan karena berpotensi menyebabkan kebocoran informasi terhadap `CVD Risk Level`.

## 3.4 Langkah Analisis

1. Memuat dan memeriksa struktur dataset asli.
2. Menghapus baris yang memiliki missing value hanya pada variabel respon atau fitur yang benar-benar dipakai model.
3. Menggunakan kolom asli dataset tanpa feature engineering.
4. Mengubah variabel kategorik menjadi dummy variable.
5. Menstandarkan seluruh prediktor numerik/dummy berdasarkan seluruh data yang digunakan membangun model.
6. Membangun model regresi logistik multinomial ridge dengan kategori referensi `LOW` menggunakan seluruh data yang lolos seleksi.
7. Mengevaluasi hasil klasifikasi pada data yang sama yang dipakai untuk membangun model.
8. Menginterpretasikan confusion matrix, akurasi, precision, recall, F1-score, koefisien, dan odds ratio.


In [1]:
import math
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

CSV_PATH = "CVD Dataset.csv"
TARGET = "CVD Risk Level"
BASELINE_CLASS = "LOW"
CLASS_ORDER = ["LOW", "INTERMEDIARY", "HIGH"]

DESCRIPTIVE_NUMERIC_PREDICTORS = [
    "Age",
    "BMI",
    "Abdominal Circumference (cm)",
    "Total Cholesterol (mg/dL)",
    "HDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Systolic BP",
    "Diastolic BP",
    "Estimated LDL (mg/dL)",
]

DESCRIPTIVE_CATEGORICAL_PREDICTORS = [
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD",
]

NUMERIC_PREDICTORS = [
    "Age",
    "BMI",
    "Abdominal Circumference (cm)",
    "Total Cholesterol (mg/dL)",
    "HDL (mg/dL)",
    "Fasting Blood Sugar (mg/dL)",
    "Systolic BP",
    "Diastolic BP",
    "Estimated LDL (mg/dL)",
    "Weight (kg)",
    "Height (m)",
    "Waist-to-Height Ratio",
]

CATEGORICAL_PREDICTORS = [
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD",
    "Blood Pressure Category",
]

PREDICTORS = NUMERIC_PREDICTORS + CATEGORICAL_PREDICTORS
BEST_L2 = 0.1


# BAB IV. Analisis dan Pembahasan

## 4.1 Memuat Data


In [2]:
df_raw = pd.read_csv(CSV_PATH)
required_columns = PREDICTORS + [TARGET]
df = df_raw.dropna(subset=required_columns).reset_index(drop=True).copy()

print("Dimensi data asli:", df_raw.shape)
print("Dimensi data setelah menghapus missing pada fitur yang dipakai:", df.shape)
print("Jumlah baris yang dihapus:", len(df_raw) - len(df))
print("Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai:", int(df["CVD Risk Score"].isna().sum()))
display(df.head())
display(pd.DataFrame({"tipe_data": df_raw.dtypes, "missing": df_raw.isna().sum()}))


Dimensi data asli: (1529, 22)
Dimensi data setelah menghapus missing pada fitur yang dipakai: (845, 22)
Jumlah baris yang dihapus: 684
Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai: 42
  Sex   Age  Weight (kg)  Height (m)   BMI  Abdominal Circumference (cm) Blood Pressure (mmHg)  Total Cholesterol (mg/dL)  HDL (mg/dL)  \
0   F  32.0         69.1        1.71  23.6                          86.2                125/79                      248.0         78.0   
1   F  55.0        118.7        1.69  41.6                          82.5                139/70                      162.0         50.0   
2   M  44.0        108.3        1.80  33.4                          96.6                140/83                      134.0         46.0   
3   F  32.0         99.5        1.86  28.8                         102.7                144/83                      146.0         64.0   
4   F  58.0        117.9        1.87  33.7                          81.4                142/90              

## 4.2 Distribusi Variabel Respon


In [3]:
target_counts = df[TARGET].value_counts().reindex(CLASS_ORDER)
target_percent = (target_counts / target_counts.sum() * 100).round(2)

target_summary = pd.DataFrame({
    "jumlah": target_counts,
    "persentase": target_percent
})
display(target_summary)


                jumlah  persentase
CVD Risk Level                    
LOW                128       15.15
INTERMEDIARY       305       36.09
HIGH               412       48.76


## 4.3 Statistik Deskriptif Prediktor Numerik


In [4]:
desc_numeric = df_raw[DESCRIPTIVE_NUMERIC_PREDICTORS].describe().T
display(desc_numeric.round(3))


                               count     mean     std    min      25%      50%      75%      max
Age                           1451.0   47.025  12.421   25.0   37.000   46.000   55.000   79.000
BMI                           1465.0   28.466   7.039   15.0   22.629   28.159   34.000   46.200
Abdominal Circumference (cm)  1462.0   91.773  12.824   70.0   80.500   91.600  102.269  119.996
Total Cholesterol (mg/dL)     1456.0  198.539  57.794  100.0  150.000  197.000  249.000  300.000
HDL (mg/dL)                   1449.0   56.197  16.067   30.0   42.000   56.000   70.000   89.000
Fasting Blood Sugar (mg/dL)   1462.0  117.486  30.289   70.0   92.000  115.000  138.000  198.000
Systolic BP                   1458.0  125.628  22.112   90.0  107.000  125.000  141.000  179.000
Diastolic BP                  1447.0   82.918  14.731   60.0   71.000   82.000   93.000  119.000
Estimated LDL (mg/dL)         1460.0  111.551  58.866  -18.0   61.000  109.000  159.000  237.000


## 4.4 Tabulasi Prediktor Kategorik


In [5]:
for col in DESCRIPTIVE_CATEGORICAL_PREDICTORS:
    print(f"\n{col}")
    display(pd.crosstab(df[col], df[TARGET], margins=True))



Sex
CVD Risk Level  HIGH  INTERMEDIARY  LOW  All
Sex                                         
F                215           162   63  440
M                197           143   65  405
All              412           305  128  845

Smoking Status
CVD Risk Level  HIGH  INTERMEDIARY  LOW  All
Smoking Status                              
N                149           188   68  405
Y                263           117   60  440
All              412           305  128  845

Diabetes Status
CVD Risk Level   HIGH  INTERMEDIARY  LOW  All
Diabetes Status                              
N                 158           177   65  400
Y                 254           128   63  445
All               412           305  128  845

Physical Activity Level
CVD Risk Level           HIGH  INTERMEDIARY  LOW  All
Physical Activity Level                              
High                      115           119   46  280
Low                       171            77   35  283
Moderate                  126           1

## 4.5 Persiapan Data


In [6]:
df_model = df[df[TARGET].isin(CLASS_ORDER)].reset_index(drop=True).copy()
y = pd.Categorical(df_model[TARGET], categories=CLASS_ORDER, ordered=True).codes

print("Jumlah seluruh data yang digunakan untuk model:", len(df_model))
display(pd.DataFrame({
    "jumlah": pd.Series(y).map(dict(enumerate(CLASS_ORDER))).value_counts().reindex(CLASS_ORDER),
    "persentase": (pd.Series(y).map(dict(enumerate(CLASS_ORDER))).value_counts().reindex(CLASS_ORDER) / len(df_model) * 100).round(2),
}))


Jumlah seluruh data yang digunakan untuk model: 845
              jumlah  persentase
LOW              128       15.15
INTERMEDIARY     305       36.09
HIGH             412       48.76


In [7]:
def preprocess_full_sample(df):
    X_raw = pd.get_dummies(
        df[PREDICTORS],
        columns=CATEGORICAL_PREDICTORS,
        drop_first=True,
        dtype=float,
    )

    means = X_raw.mean()
    stds = X_raw.std(ddof=0).replace(0, 1)
    X_raw = (X_raw - means) / stds

    X = np.column_stack([np.ones(len(X_raw)), X_raw.to_numpy(float)])
    feature_names = ["Intercept"] + X_raw.columns.tolist()
    return X, feature_names, means, stds

X_all, feature_names, means, stds = preprocess_full_sample(df_model)
y_all = y.copy()

print("Jumlah fitur termasuk intercept:", X_all.shape[1])
print("Regularisasi ridge terbaik yang digunakan:", BEST_L2)
print("Nama fitur:")
display(pd.Series(feature_names))


Jumlah fitur termasuk intercept: 22
Regularisasi ridge terbaik yang digunakan: 0.1
Nama fitur:
0                                        Intercept
1                                              Age
2                                              BMI
3                     Abdominal Circumference (cm)
4                        Total Cholesterol (mg/dL)
5                                      HDL (mg/dL)
6                      Fasting Blood Sugar (mg/dL)
7                                      Systolic BP
8                                     Diastolic BP
9                            Estimated LDL (mg/dL)
10                                     Weight (kg)
11                                      Height (m)
12                           Waist-to-Height Ratio
13                                           Sex_M
14                                Smoking Status_Y
15                               Diabetes Status_Y
16                     Physical Activity Level_Low
17                Physical Activity Le

## 4.6 Pemodelan Regresi Logistik Multinomial

Model di bawah dibuat dengan parameterisasi baseline-category logit. Kategori referensi adalah `LOW`, sehingga terdapat dua persamaan logit:

- `INTERMEDIARY` dibandingkan `LOW`
- `HIGH` dibandingkan `LOW`

Pada versi utama ini, model dibangun secara lebih sederhana dan transparan:

- Hanya memakai kolom asli dataset tanpa feature engineering.
- Baris dihapus hanya jika target atau fitur yang benar-benar dipakai mengandung missing value.
- `CVD Risk Score` tetap tidak digunakan karena berpotensi langsung menentukan `CVD Risk Level`.
- Menggunakan ridge regularization kecil (`l2 = 0.1`) untuk menjaga kestabilan estimasi.


In [8]:
def softmax_baseline(X, B):
    eta = X @ B
    scores = np.column_stack([np.zeros(X.shape[0]), eta])
    scores -= scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

def neg_loglik(X, y, B, l2=1e-6):
    P = softmax_baseline(X, B)
    eps = 1e-15
    penalty = 0.5 * l2 * np.sum(B[1:, :] ** 2)
    return -np.log(P[np.arange(len(y)), y] + eps).sum() + penalty

def hessian_baseline(X, P, l2=1e-6):
    n, p = X.shape
    K = P.shape[1]
    H = np.zeros((p * (K - 1), p * (K - 1)))
    for a in range(K - 1):
        for b in range(K - 1):
            pa = P[:, a + 1]
            pb = P[:, b + 1]
            w = pa * ((1 if a == b else 0) - pb)
            block = X.T @ (X * w[:, None])
            if a == b:
                reg = np.eye(p) * l2
                reg[0, 0] = 0
                block += reg
            H[a*p:(a+1)*p, b*p:(b+1)*p] = block
    return H

def fit_multinomial_logit(X, y, max_iter=80, tol=1e-7, l2=1e-6):
    n, p = X.shape
    classes = np.unique(y)
    K = len(classes)
    if not np.array_equal(classes, np.arange(K)):
        raise ValueError("y harus dikodekan 0 sampai K-1.")

    B = np.zeros((p, K - 1))
    Y = np.eye(K)[y][:, 1:]
    history = []

    for iteration in range(1, max_iter + 1):
        P = softmax_baseline(X, B)
        gradient = X.T @ (P[:, 1:] - Y)
        gradient[1:, :] += l2 * B[1:, :]
        H = hessian_baseline(X, P, l2=l2)

        grad_flat = gradient.T.reshape(-1)
        try:
            step_flat = np.linalg.solve(H, grad_flat)
        except np.linalg.LinAlgError:
            step_flat = np.linalg.pinv(H) @ grad_flat

        step = step_flat.reshape(K - 1, p).T
        current_loss = neg_loglik(X, y, B, l2=l2)
        step_scale = 1.0

        while step_scale > 1e-6:
            candidate = B - step_scale * step
            candidate_loss = neg_loglik(X, y, candidate, l2=l2)
            if candidate_loss <= current_loss:
                break
            step_scale *= 0.5

        B = candidate
        history.append(candidate_loss)

        if np.linalg.norm(step_scale * step) < tol:
            break

    P_final = softmax_baseline(X, B)
    H_final = hessian_baseline(X, P_final, l2=l2)
    try:
        covariance = np.linalg.inv(H_final)
    except np.linalg.LinAlgError:
        covariance = np.linalg.pinv(H_final)

    info = {
        "iterations": iteration,
        "loss": history[-1],
        "history": history,
        "covariance": covariance,
    }
    return B, info

def predict_multinomial(X, B):
    probabilities = softmax_baseline(X, B)
    predicted_class = probabilities.argmax(axis=1)
    return predicted_class, probabilities


In [9]:
B, fit_info = fit_multinomial_logit(X_all, y_all, max_iter=80, tol=1e-7, l2=BEST_L2)

print("Iterasi konvergensi:", fit_info["iterations"])
print("Negative log-likelihood akhir:", round(float(fit_info["loss"]), 4))
print("L2 regularization:", BEST_L2)


Iterasi konvergensi: 6
Negative log-likelihood akhir: 701.9092
L2 regularization: 0.1


## 4.7 Evaluasi Model


In [10]:
y_pred, y_prob = predict_multinomial(X_all, B)
class_map = dict(enumerate(CLASS_ORDER))

accuracy = (y_pred == y_all).mean()
print("Akurasi pada seluruh data pembentukan model:", round(float(accuracy), 4))

confusion = pd.crosstab(
    pd.Series(y_all).map(class_map),
    pd.Series(y_pred).map(class_map),
    rownames=["Aktual"],
    colnames=["Prediksi"],
).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)

display(confusion)


Akurasi pada seluruh data pembentukan model: 0.6828
Prediksi      LOW  INTERMEDIARY  HIGH
Aktual                               
LOW            19            58    51
INTERMEDIARY    9           212    84
HIGH           11            55   346


In [11]:
def classification_report_manual(y_true, y_pred, labels):
    rows = []
    for i, label in enumerate(labels):
        tp = np.sum((y_true == i) & (y_pred == i))
        fp = np.sum((y_true != i) & (y_pred == i))
        fn = np.sum((y_true == i) & (y_pred != i))
        support = np.sum(y_true == i)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append([label, precision, recall, f1, support])
    return pd.DataFrame(rows, columns=["kelas", "precision", "recall", "f1_score", "support"])

report = classification_report_manual(y_all, y_pred, CLASS_ORDER)
display(report.round(4))


          kelas  precision  recall  f1_score  support
0           LOW     0.4872  0.1484    0.2275      128
1  INTERMEDIARY     0.6523  0.6951    0.6730      305
2          HIGH     0.7193  0.8398    0.7749      412


## 4.8 Koefisien, Nilai p Pendekatan, dan Odds Ratio


In [12]:
coef = pd.DataFrame(B, index=feature_names, columns=CLASS_ORDER[1:])

p = X_all.shape[1]
K_minus_1 = len(CLASS_ORDER) - 1
covariance = fit_info["covariance"]
se_flat = np.sqrt(np.maximum(np.diag(covariance), 0))
se = se_flat.reshape(K_minus_1, p).T
se_df = pd.DataFrame(se, index=feature_names, columns=CLASS_ORDER[1:])

z_df = coef / se_df.replace(0, np.nan)
erfc_vec = np.vectorize(math.erfc)
pvalue_df = pd.DataFrame(
    erfc_vec(np.abs(z_df.to_numpy(float)) / math.sqrt(2)),
    index=feature_names,
    columns=CLASS_ORDER[1:],
)

odds_ratio = np.exp(coef)

coef_table = (
    coef.stack().rename("coef")
    .to_frame()
    .join(se_df.stack().rename("std_error"))
    .join(z_df.stack().rename("z_value"))
    .join(pvalue_df.stack().rename("p_value"))
    .join(odds_ratio.stack().rename("odds_ratio"))
    .reset_index()
    .rename(columns={"level_0": "variabel", "level_1": "kategori_vs_LOW"})
)

display(coef_table.round(4))


                                        variabel kategori_vs_LOW    coef  std_error  z_value  p_value  odds_ratio
0                                      Intercept    INTERMEDIARY  0.9064     0.1269   7.1417   0.0000      2.4755
1                                      Intercept            HIGH  1.2220     0.1216  10.0528   0.0000      3.3941
2                                            Age    INTERMEDIARY -0.2629     0.1061  -2.4782   0.0132      0.7688
3                                            Age            HIGH  0.0956     0.1072   0.8919   0.3725      1.1003
4                                            BMI    INTERMEDIARY -0.3701     0.1422  -2.6020   0.0093      0.6907
5                                            BMI            HIGH  0.0778     0.1419   0.5478   0.5838      1.0809
6                   Abdominal Circumference (cm)    INTERMEDIARY  1.3915     1.0899   1.2767   0.2017      4.0210
7                   Abdominal Circumference (cm)            HIGH -0.6796     1.1043  -0.

## 4.9 Variabel dengan Odds Ratio Terbesar


In [13]:
for category in CLASS_ORDER[1:]:
    print(f"\nOdds ratio terbesar untuk {category} dibandingkan LOW")
    temp = odds_ratio[category].drop("Intercept").sort_values(ascending=False)
    display(temp.head(10).to_frame("odds_ratio").round(4))



Odds ratio terbesar untuk INTERMEDIARY dibandingkan LOW
                                              odds_ratio
Abdominal Circumference (cm)                      4.0210
Weight (kg)                                       1.3216
Blood Pressure Category_Hypertension Stage 1      1.0893
Family History of CVD_Y                           1.0394
Estimated LDL (mg/dL)                             0.9656
Physical Activity Level_Moderate                  0.9612
Total Cholesterol (mg/dL)                         0.9307
Physical Activity Level_Low                       0.9171
Sex_M                                             0.9061
Fasting Blood Sugar (mg/dL)                       0.8905

Odds ratio terbesar untuk HIGH dibandingkan LOW
                                              odds_ratio
Waist-to-Height Ratio                             2.2564
Family History of CVD_Y                           1.6126
Physical Activity Level_Low                       1.6057
Smoking Status_Y                       

## 4.10 Interpretasi Singkat

Cara membaca output:

- Jika odds ratio suatu variabel lebih besar dari 1 pada kolom `HIGH`, maka kenaikan variabel tersebut berasosiasi dengan meningkatnya odds pasien berada pada risiko `HIGH` dibandingkan `LOW`, dengan asumsi variabel lain konstan.
- Jika odds ratio kurang dari 1, maka variabel tersebut berasosiasi dengan menurunnya odds kategori tersebut dibandingkan `LOW`.
- Untuk prediktor numerik, interpretasi berlaku setelah standardisasi. Artinya, perubahan satu satuan adalah satu standar deviasi pada seluruh data complete-case yang dipakai membangun model.
- Untuk prediktor kategorik dummy, interpretasi dibandingkan kategori referensi yang otomatis di-drop saat one-hot encoding.

Catatan penting: interpretasi ini adalah asosiasi statistik, bukan bukti kausal.


In [14]:
interpretation_rows = []
for category in CLASS_ORDER[1:]:
    temp = odds_ratio[category].drop("Intercept").sort_values(ascending=False)
    top_increase = temp.head(5)
    top_decrease = temp.tail(5).sort_values()
    for variable, value in top_increase.items():
        interpretation_rows.append([category, "meningkatkan odds", variable, value])
    for variable, value in top_decrease.items():
        interpretation_rows.append([category, "menurunkan odds", variable, value])

interpretation_table = pd.DataFrame(
    interpretation_rows,
    columns=["kategori_vs_LOW", "arah_asosiasi", "variabel", "odds_ratio"]
)
display(interpretation_table.round(4))


   kategori_vs_LOW      arah_asosiasi                                      variabel  odds_ratio
0     INTERMEDIARY  meningkatkan odds                  Abdominal Circumference (cm)      4.0210
1     INTERMEDIARY  meningkatkan odds                                   Weight (kg)      1.3216
2     INTERMEDIARY  meningkatkan odds  Blood Pressure Category_Hypertension Stage 1      1.0893
3     INTERMEDIARY  meningkatkan odds                       Family History of CVD_Y      1.0394
4     INTERMEDIARY  meningkatkan odds                         Estimated LDL (mg/dL)      0.9656
5     INTERMEDIARY    menurunkan odds                         Waist-to-Height Ratio      0.2566
6     INTERMEDIARY    menurunkan odds                                    Height (m)      0.6178
7     INTERMEDIARY    menurunkan odds                                           BMI      0.6907
8     INTERMEDIARY    menurunkan odds                                           Age      0.7688
9     INTERMEDIARY    menurunkan odds   

## 4.11 Catatan Evaluasi

Pada versi notebook ini, model regresi logistik multinomial dibangun menggunakan **seluruh 845 data** yang memiliki kelengkapan pada target dan fitur yang dipakai. Karena itu, angka akurasi yang dilaporkan di atas adalah **akurasi pada data pembentukan model** (apparent accuracy), bukan akurasi generalisasi pada data baru.

Pilihan ini sesuai dengan permintaan untuk menggunakan seluruh data dalam pembentukan model. Namun secara metodologis, konsekuensinya adalah performa model pada data baru tidak dapat dievaluasi secara terpisah pada versi notebook ini.


In [15]:
print("Bagian perbandingan model non-logistik tidak dijalankan pada versi utama tanpa feature engineering ini.")

Bagian perbandingan model non-logistik tidak dijalankan pada versi utama tanpa feature engineering ini.


**Catatan:** pada versi utama ini, fokus analisis diarahkan pada model regresi logistik multinomial tanpa feature engineering. Perbandingan dengan model lain tidak dijalankan agar alur notebook tetap konsisten dengan model utama yang dipilih.


## 4.12 Implikasi Pemakaian Seluruh Data

Pemakaian seluruh data yang lolos seleksi kelengkapan fitur memberikan manfaat berupa penggunaan semua informasi yang relevan dalam estimasi koefisien. Akan tetapi, akurasi yang diperoleh menjadi akurasi pada data yang sama dengan data pembentukan model, sehingga tidak boleh diartikan sebagai kemampuan prediksi pada data baru.

Dengan kata lain, versi notebook ini lebih tepat dipakai untuk:

- membentuk model akhir pada seluruh data yang memenuhi syarat,
- mengestimasi koefisien dan odds ratio,
- mendeskripsikan pola asosiasi antar faktor risiko dan kategori `CVD Risk Level`.

Versi ini kurang tepat bila tujuan utamanya adalah menilai performa generalisasi model.


## 4.13 Ringkasan Data yang Dipakai Model

Model utama pada notebook ini sekarang menggunakan hanya baris yang lengkap pada variabel respon dan fitur asli yang dipakai. Dengan aturan ini, baris dengan `CVD Risk Score` kosong tetap boleh dipertahankan karena variabel tersebut tidak menjadi prediktor model.


In [16]:
complete_case_summary = pd.DataFrame({
    "jumlah": df[TARGET].value_counts().reindex(CLASS_ORDER),
    "persentase": (df[TARGET].value_counts().reindex(CLASS_ORDER) / len(df) * 100).round(2),
})

print("Baris awal:", len(df_raw))
print("Baris yang dipakai model:", len(df))
print("Baris yang dihapus:", len(df_raw) - len(df))
print("Proporsi data yang dipertahankan (%):", round(len(df) / len(df_raw) * 100, 2))
print("Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai:", int(df["CVD Risk Score"].isna().sum()))
display(complete_case_summary)


Baris awal: 1529
Baris yang dipakai model: 845
Baris yang dihapus: 684
Proporsi data yang dipertahankan (%): 55.26
Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai: 42
                jumlah  persentase
CVD Risk Level                    
LOW                128       15.15
INTERMEDIARY       305       36.09
HIGH               412       48.76


**Interpretasi data utama:** setelah hanya menghapus missing value pada target dan fitur yang dipakai, ukuran data yang digunakan menjadi 845 pasien. Pendekatan ini mempertahankan lebih banyak observasi daripada aturan complete-case seluruh kolom. Distribusi kelas tetap menunjukkan `HIGH` sebagai kelas dominan, `INTERMEDIARY` di tengah, dan `LOW` sebagai kelas minoritas.


## 4.14 Catatan Model Utama

Model utama notebook ini sekarang adalah regresi logistik multinomial yang memakai **kolom asli dataset tanpa feature engineering**. Dengan pendekatan ini, model menjadi lebih sederhana, lebih mudah dijelaskan, dan tetap mempertahankan lebih banyak baris data selama fitur yang dipakai lengkap.


In [17]:
print("Pendekatan tanpa feature engineering sudah dijadikan model utama notebook ini.")
print("Ringkasan hasil utama:")
print("- Jumlah baris yang dipakai: 845")
print("- Jumlah baris yang dihapus: 684")
print("- Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai: 42")
print("- Akurasi pada data pembentukan model: 0.6828")


Pendekatan tanpa feature engineering sudah dijadikan model utama notebook ini.
Ringkasan hasil utama:
- Jumlah baris yang dipakai: 845
- Jumlah baris yang dihapus: 684
- Jumlah baris dengan CVD Risk Score kosong tetapi tetap dipakai: 42
- Akurasi pada data pembentukan model: 0.6828


**Catatan:** hasil tanpa feature engineering kini menjadi hasil utama yang dipakai di seluruh notebook.


# BAB V. Kesimpulan dan Saran

## 5.1 Kesimpulan

1. Dataset CAIR-CVD-2025 memiliki 1.529 observasi pasien dengan tiga kategori tingkat risiko CVD, yaitu `LOW`, `INTERMEDIARY`, dan `HIGH`.
2. Regresi logistik multinomial sesuai digunakan karena variabel respon memiliki lebih dari dua kategori.
3. Model utama dibangun menggunakan kolom asli dataset tanpa feature engineering dan tanpa `CVD Risk Score` sebagai prediktor.
4. Setelah hanya menghapus baris yang missing pada target atau fitur yang dipakai, model menggunakan 845 observasi.
5. Model akhir tetap menggunakan kategori `LOW` sebagai referensi, sehingga koefisien dan odds ratio menginterpretasikan perbandingan `INTERMEDIARY` vs `LOW` dan `HIGH` vs `LOW`.
6. Evaluasi model dilakukan menggunakan confusion matrix, akurasi, precision, recall, dan F1-score pada data pembentukan model.
7. Odds ratio membantu mengidentifikasi prediktor yang paling kuat berasosiasi dengan kenaikan atau penurunan odds risiko CVD.

## 5.2 Saran

1. Analisis lanjutan dapat membandingkan hasil dengan ordinal logistic regression karena level risiko memiliki urutan alami.
2. Validasi eksternal pada data dari rumah sakit atau wilayah lain diperlukan sebelum model digunakan sebagai alat pendukung keputusan.
3. Variabel `CVD Risk Score` sebaiknya tetap tidak digunakan dalam model prediksi utama karena dapat menyebabkan information leakage jika digunakan untuk memprediksi `CVD Risk Level`.


# Lampiran: Prediksi Probabilitas Data Uji


In [18]:
probability_table = pd.DataFrame(y_prob, columns=[f"prob_{label}" for label in CLASS_ORDER])
probability_table.insert(0, "aktual", pd.Series(y_all).map(class_map).to_numpy())
probability_table.insert(1, "prediksi", pd.Series(y_pred).map(class_map).to_numpy())
display(probability_table.head(20).round(4))


          aktual      prediksi  prob_LOW  prob_INTERMEDIARY  prob_HIGH
0   INTERMEDIARY  INTERMEDIARY    0.1568             0.5175     0.3258
1           HIGH          HIGH    0.0382             0.0774     0.8844
2   INTERMEDIARY  INTERMEDIARY    0.0854             0.6383     0.2763
3   INTERMEDIARY  INTERMEDIARY    0.1263             0.6101     0.2636
4           HIGH          HIGH    0.1732             0.2892     0.5376
5   INTERMEDIARY          HIGH    0.0757             0.2797     0.6446
6           HIGH          HIGH    0.0917             0.1862     0.7222
7   INTERMEDIARY  INTERMEDIARY    0.1912             0.5201     0.2888
8           HIGH          HIGH    0.0299             0.0558     0.9144
9   INTERMEDIARY  INTERMEDIARY    0.1508             0.5405     0.3086
10          HIGH          HIGH    0.1095             0.1356     0.7548
11  INTERMEDIARY          HIGH    0.1107             0.3824     0.5069
12          HIGH          HIGH    0.0461             0.1001     0.8538
13    